In [0]:
# =========================================================
# SERVERLESS-SAFE COMPARISON
# Baseline vs Logistic Regression
# Run in a fresh session
# =========================================================

import gc
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# ---------------------------------------------------------
# USER SETTINGS
# ---------------------------------------------------------
CITY = "Toronto"   # "Toronto" or "NYC"
SEED = 42
LABEL = "delay_indicator"

# ---------------------------------------------------------
# CLEANUP
# ---------------------------------------------------------
for v in [
    "df", "train_df", "test_df", "baseline_pred", "pred_df",
    "pipeline", "model", "assembler", "results_df"
]:
    if v in globals():
        del globals()[v]
gc.collect()

try:
    spark.catalog.clearCache()
except:
    pass

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
table_name = (
    "workspace.capstone_project.toronto_model_ready"
    if CITY == "Toronto"
    else "workspace.capstone_project.nyc_model_ready"
)

candidate_cols = [
    LABEL,
    "hour", "day_of_week", "month", "year",
    "unified_alarm_level", "calls_past_30min", "calls_past_60min",
    "season", "incident_category", "unified_call_source", "location_area"
]

df = spark.table(table_name)
keep_cols = [c for c in candidate_cols if c in df.columns]
df = df.select(*keep_cols).filter(col(LABEL).isNotNull())

print(f"City: {CITY}")
print(f"Rows: {df.count()}")
df.groupBy(LABEL).count().orderBy(LABEL).show()

# ---------------------------------------------------------
# TRAIN / TEST SPLIT
# ---------------------------------------------------------
train_df, test_df = df.randomSplit([0.8, 0.2], seed=SEED)

print("Train count:", train_df.count())
print("Test count :", test_df.count())

# ---------------------------------------------------------
# EVALUATION FUNCTION
# ---------------------------------------------------------
def evaluate_predictions(pred_df, model_name):
    total = pred_df.count()

    accuracy = pred_df.filter(col(LABEL) == col("prediction")).count() / total

    precision = MulticlassClassificationEvaluator(
        labelCol=LABEL,
        predictionCol="prediction",
        metricName="weightedPrecision"
    ).evaluate(pred_df)

    recall = MulticlassClassificationEvaluator(
        labelCol=LABEL,
        predictionCol="prediction",
        metricName="weightedRecall"
    ).evaluate(pred_df)

    f1 = MulticlassClassificationEvaluator(
        labelCol=LABEL,
        predictionCol="prediction",
        metricName="f1"
    ).evaluate(pred_df)

    if "rawPrediction" in pred_df.columns:
        auc = BinaryClassificationEvaluator(
            labelCol=LABEL,
            rawPredictionCol="rawPrediction",
            metricName="areaUnderROC"
        ).evaluate(pred_df)
    else:
        auc = 0.5

    print("=====================================================")
    print(f"{CITY} - {model_name}")
    print("=====================================================")
    print(f"Accuracy : {accuracy:.6f}")
    print(f"AUC-ROC  : {auc:.6f}")
    print(f"Precision: {precision:.6f}")
    print(f"Recall   : {recall:.6f}")
    print(f"F1 Score : {f1:.6f}")

    pred_df.groupBy(LABEL, "prediction").count().orderBy(LABEL, "prediction").show()

    return (CITY, model_name, float(accuracy), float(auc), float(precision), float(recall), float(f1))

# ---------------------------------------------------------
# 1. BASELINE
# ---------------------------------------------------------
majority = (
    train_df.groupBy(LABEL)
    .count()
    .orderBy(F.desc("count"))
    .first()[LABEL]
)

baseline_pred = test_df.withColumn("prediction", F.lit(float(majority)))
baseline_result = evaluate_predictions(baseline_pred, "Naive Majority Baseline")

del baseline_pred
gc.collect()

# ---------------------------------------------------------
# 2. LOGISTIC REGRESSION
# ---------------------------------------------------------
numeric_cols = [c for c in [
    "hour", "day_of_week", "month", "year",
    "unified_alarm_level", "calls_past_30min", "calls_past_60min"
] if c in df.columns]

categorical_cols = [c for c in [
    "season", "incident_category", "unified_call_source", "location_area"
] if c in df.columns]

stages = []
encoded_cols = []

for c in categorical_cols:
    idx = f"{c}_idx"
    ohe = f"{c}_ohe"
    stages.append(StringIndexer(inputCol=c, outputCol=idx, handleInvalid="keep"))
    stages.append(OneHotEncoder(inputCols=[idx], outputCols=[ohe]))
    encoded_cols.append(ohe)

assembler = VectorAssembler(
    inputCols=numeric_cols + encoded_cols,
    outputCol="features",
    handleInvalid="skip"
)

lr = LogisticRegression(
    labelCol=LABEL,
    featuresCol="features",
    predictionCol="prediction",
    rawPredictionCol="rawPrediction",
    probabilityCol="probability",
    maxIter=20,
    regParam=0.01,
    elasticNetParam=0.0
)

pipeline = Pipeline(stages=stages + [assembler, lr])
model = pipeline.fit(train_df)

pred_df = model.transform(test_df).select(LABEL, "prediction", "rawPrediction")
lr_result = evaluate_predictions(pred_df, "Logistic Regression")

# ---------------------------------------------------------
# 3. FINAL COMPARISON MATRIX
# ---------------------------------------------------------
results_df = spark.createDataFrame(
    [baseline_result, lr_result],
    ["City", "Model", "Accuracy", "AUC_ROC", "Precision", "Recall", "F1_Score"]
)

results_matrix = (
    results_df
    .withColumn("Accuracy", F.round("Accuracy", 3))
    .withColumn("AUC_ROC", F.round("AUC_ROC", 3))
    .withColumn("Precision", F.round("Precision", 3))
    .withColumn("Recall", F.round("Recall", 3))
    .withColumn("F1_Score", F.round("F1_Score", 3))
)

print("=====================================================")
print("Baseline vs Logistic Regression Comparison")
print("=====================================================")
results_matrix.show(truncate=False)

# ---------------------------------------------------------
# 4. CLEANUP
# ---------------------------------------------------------
for v in [
    "pred_df", "model", "pipeline", "assembler", "lr",
    "train_df", "test_df", "df", "results_df"
]:
    if v in globals():
        del globals()[v]

gc.collect()

try:
    spark.catalog.clearCache()
except:
    pass

print("Done.")